> Globals:
> - mnist
> - X
> - y
>- X_direct, 
> - y_direct
> - y_predicted
> - model

> Imports for all examples.

In [ ]:
import numpy as np
import pandas as pd

from sklearn.datasets import fetch_openml
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler, MinMaxScaler, PolynomialFeatures
from sklearn.decomposition import PCA
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, classification_report, 
    ConfusionMatrixDisplay
)
from sklearn.pipeline import make_pipeline

import joblib  #dump, load

import matplotlib.pyplot as plt

> Increases the number of columns that get shown in a table, allowing data in the middle of the Series to be seen.

In [ ]:
pd.set_option("display.max_columns", None)

> Imports the MNIST dataset. 
>
> See OpenML for more:  
> https://www.openml.org/d/554

In [ ]:
mnist = fetch_openml("mnist_784", version=1)

> The next few cells show what's contained in the dataset.  
> The image data can be accessed via the "data" key. This is 784 integers representing the pixels of a 28x28 image.  
> The actual digits those images represent can be accessed via the "target" key. 
>
> Note that the data is a Pandas DataFrame and target is a Pandas Series, representing a matrix, and a vector, respectively.    
> Scikit-learn usually denotes data with a capital letter, and labels without.

In [ ]:
print(mnist.keys())

In [ ]:
X = mnist["data"]
y = mnist["target"]

> Raw data: 
> - 70000 samples 
> - 784 features - 28 pixels x 28 pixels
>
> Horizontally scroll to about pixel177 to see some values.

In [ ]:
X

In [ ]:
y

In [ ]:
print("Data:")
print(type(X))
print(type(X.iloc[0]))
print(type(X.iloc[0].iloc[0]))
print(X.shape)
print()

print("Target:")
print(type(y))
print(type(y.iloc[0]))
print(y.shape)

In [ ]:
print ("First series:") 
print(X.iloc[0])  # Series

print()

print("Value of a specific feature in the first series:")
print(X.iloc[0]["pixel184"])  # Series value for feature "pixel184"

print()

print("First target:")
print(y[0])  # Scalar

> The dataset can also be imported, without the overhead of a DataFrame, directly into X and y
>
> Note that without a DataFrame these are a pure matix (not a DataFrame) and a pure vector (not a Series).
> Therefore, you can only get to the value of a feature, e.g. "pixel184", via an index knowing that labels follow
> the format "pixel1" = [1] etc.

In [ ]:
X_direct, y_direct = fetch_openml("mnist_784", version=1, return_X_y=True, as_frame=False)

In [ ]:
print("Data:")
print(type(X_direct))
print(type(X_direct[0]))
print(type(X_direct[0][0]))
print(X_direct.shape)
print()

print("Target:")
print(type(y_direct))
print(type(y_direct[0]))
print(y_direct.shape)
print()

print ("First vector:") 
print(X_direct[0])
print()

print("Value of a specific feature in the first vector:")
print(X_direct[0][184])
print()

print("First target:")
print(y_direct[0])

> The X (DataFrame) and y (Series) values will be used from here on, as the extra Pandas functionality makes it easier
> for visualizations.

> Converts the target values from string to int to make it easier to use in all future examples.

In [ ]:
y = y.astype(int)

> Takes the data (X), and target (y) and splits them into two parts. 
> One is used to train the model (60000 samples), and the other is used to see how accurate that training 
> was (10000 samples).
>
> Note that many examples show splitting data like this:
> ```python
> X_train, X_test, y_train, y_test = train_test_split(data, target, test_size=10000, random_state=42)
> ```
> This is fine but is not appropriate for comparisons. In order to be able to compare with benchmarks you must use
> the same 10000 samples used by those implementations.

In [ ]:

X_train, X_test = X[:60000], X[60000:]
y_train, y_test = y[:60000], y[60000:]

> Converts the first 10 samples to 28x28 arrays, and displays them as images.

In [ ]:
def plot_first_ten_images():
    _, axes = plt.subplots(2, 5, figsize=(6, 3), constrained_layout=True)

    for index, axis in enumerate(axes.flat):
        axis.imshow(X.iloc[index].values.reshape(28, 28), cmap=plt.cm.Greys)
        axis.set_title(f"target: {y[index]}")
        axis.axis("off")

    plt.show()

plot_first_ten_images()

> Visualizes the Sigmoid funtion. Used to determine the probability of a match. This produces a number between 0 and 1 
> that indicates how well the sample fits the model. For example, 0.5 or above is a match. Below 0.5 is not a match. 
> For MNSIT there are 10 classes so this yes/no is not used, and the highest match is picked. 

In [ ]:
def sigmoid(x): return 1 / (1 + np.exp(-x))

In [ ]:
def plot_sigmoid():
    x_plot = np.linspace(-10, 10, 100)
    y_plot = sigmoid(x_plot)

    plt.figure(figsize=(6, 3), constrained_layout=True)
    plt.plot(x_plot, y_plot)
    plt.axhline(y=0.5, linestyle='--', color=plt.cm.Reds(0.5))
    plt.title("Sigmoid Function")
    plt.xlabel("x")
    plt.ylabel("sigmoid(x)")
    plt.grid()

    plt.show()

plot_sigmoid()

In [ ]:
def plot_simulated_logistic_regression():
    x_plot = np.linspace(-10, 10, 100)
    y_plot = sigmoid(x_plot)

    plt.figure(figsize=(6, 3), constrained_layout=True)
    plt.plot(x_plot, y_plot)
    plt.title("Simulated Logistic Regression with Sigmoid Function")
    plt.xlabel("x")
    plt.ylabel("sigmoid(x)")
    plt.grid()

    # Simulate logistic regression

    samples = 10
    features = 5

    weights = np.random.randn(features)
    bias = np.random.randn()

    x_random = np.random.randn(samples, features)
    x_point = x_random @ weights + bias 
    y_point = sigmoid(x_point)

    plt.scatter(x_point, y_point, color=plt.cm.Greens(0.5))

    plt.show()

plot_simulated_logistic_regression()

As an aside:
> There are other functions, ReLU, tanh, and softmax, that are also associated with deep learning, primarily as 
> activation functions in neural networks.
>
> **N.B. While these may not be appropriate for this example, they're shown here simply out of curiosity.**

In [ ]:
def relu(x): return np.maximum(0, x)

def tanh(x): return (np.exp(x) - np.exp(-x)) / (np.exp(x) + np.exp(-x))

def softmax(x):
    x = np.array(x, dtype=float)
    x_shifted = x - np.max(x)      # For numerical stability
    exp_x = np.exp(x_shifted)
    
    return exp_x / np.sum(exp_x)

In [ ]:
def plot_relu():
    x_plot = np.linspace(-10, 10, 100)
    y_plot = relu(x_plot)

    plt.figure(figsize=(6, 3), constrained_layout=True)
    plt.plot(x_plot, y_plot)
    plt.title("ReLU Function")
    plt.xlabel("x")
    plt.ylabel("ReLU(x)")
    plt.grid()

    plt.show()

plot_relu()

In [ ]:
def plot_tanh(): 
    x_plot = np.linspace(-10, 10, 100)
    y_plot = tanh(x_plot)

    plt.figure(figsize=(6, 3), constrained_layout=True)
    plt.plot(x_plot, y_plot)
    plt.title("tanh Function")
    plt.xlabel("x")
    plt.ylabel("tanh(x)")
    plt.grid()

    plt.show()

plot_tanh()

In [ ]:
def display_softmax_distribution():
    values = np.array([1, 4, 2, 3])
    result = softmax(values)

    print(result)
    print("Sum:", result.sum())  # Should be 1

display_softmax_distribution()

In [ ]:
def plot_softmax_distribution():
    x_point = np.array([1, 2, 3, 4, 5, 6, 7])
    y_point = softmax(x_point)

    plt.figure(figsize=(6, 3), constrained_layout=True)
    plt.bar(x_point, y_point, color=plt.cm.Greens(0.5))
    plt.xlim(0, 8)
    plt.ylim(0, 1)
    plt.title("Softmax Function Distribution")
    plt.xlabel("x")
    plt.ylabel("softmax(x)")
    plt.grid()

    plt.show()

plot_softmax_distribution()

> Train the model using Logical Regression with the 60000 samples, and predict using the 10000 samples.
>
> Note that this code fails to converge after 1000 iterations, so accuracy is not as good as it could be. This will be 
> improved later.

In [ ]:
def fit_and_predict_with_logistic_regrassion(X_train, y_train, X_test):
    # Train
    model = LogisticRegression(max_iter=1000)
    model.fit(X_train, y_train)

    # Test
    y_predicted = model.predict(X_test)

    return y_predicted

In [ ]:
y_predicted = fit_and_predict_with_logistic_regrassion(X_train, y_train, X_test);

> Shows the first 10 images with the actual digit, and the predicted digit.

In [ ]:
def plot_first_ten_predictions(X_test, y_predicted):
    _, axes = plt.subplots(2, 5, figsize=(6, 3), constrained_layout=True)

    for index, axis in enumerate(axes.flat):
        axis.imshow(X_test.iloc[index].values.reshape(28, 28), cmap=plt.cm.Greys_r)
        axis.set_title(f"target: {y_test.iloc[index]}\nPredicted: {y_predicted[index]}")
        axis.axis("off")

    plt.show()

    accuracy = accuracy_score(y_test, y_predicted)
    print(f"Accuracy: {accuracy * 100:.2f}%")

plot_first_ten_predictions(X_test, y_predicted)

> Show some misclassified images, so you can see where some of the confusion comes from.  

In [ ]:
def plot_misclassified_images(train_len, X_test, y_test, y_predicted):
    misclassified = np.where(y_test != y_predicted)[0]

    _, axes = plt.subplots(2, 5, figsize=(6, 3), constrained_layout=True)

    for index, axis in enumerate(axes.flat):
        misclassified_index = misclassified[index]
        axis.imshow(X_test.iloc[misclassified_index].values.reshape(28, 28), cmap=plt.cm.Grays_r)
        axis.set_title(f"target: {y_test.iloc[misclassified_index]}\nPredicted: {y_predicted[misclassified_index]}")
        axis.axis("off")

    plt.show()

    print(f"Trained: {train_len}")
    print(f"Tested: {len(y_test)} ({len(y_test) / len(y_train) * 100:.2f}%)")
    print(f"Misclassified: {misclassified.size} ({misclassified.size / len(y_test) * 100:.2f}%)")

In [ ]:
plot_misclassified_images(len(X_train), X_test, y_test, y_predicted)

> A confusion matrix shows actual vs predicted, with the diagonal showing the number of correctly predicted digits. 

In [ ]:
def plot_manual_confusion_matrix(y_test, y_predicted):
    cm = confusion_matrix(y_test, y_predicted)

    plt.figure(figsize=(10, 5), constrained_layout=True)
    plt.imshow(cm, interpolation="nearest", cmap=plt.cm.Blues)

    plt.title("Confusion Matrix")
    plt.xlabel("Predicted")
    plt.ylabel("Actual")

    tick_marks = range(len(set(y_test)))
    plt.xticks(tick_marks)
    plt.yticks(tick_marks)

    for actual in range(cm.shape[0]):        
        for predicted in range(cm.shape[1]):
            plt.text(predicted, actual, format(cm[actual, predicted], "d"), 
                    horizontalalignment="center", verticalalignment="center",
                    color="white" if cm[actual, predicted] > cm.max() / 2 else "black")

    plt.show()

plot_manual_confusion_matrix(y_test, y_predicted)

> Rather than manually coding the confusion matrix, there's a built in ```ConfusionMatrixDisplay```

In [ ]:
def plot_confusion_matrix(y_test, y_predicted):
    cm = confusion_matrix(y_test, y_predicted)

    _, ax = plt.subplots(figsize=(10, 5), constrained_layout=True)

    disp = ConfusionMatrixDisplay(confusion_matrix=cm)
    disp.plot(ax=ax, colorbar=False, cmap=plt.cm.Blues)

    plt.colorbar=False
    plt.title('Confusion Matrix')
    plt.xlabel("Predicted")
    plt.ylabel("Actual")
    
    plt.show()

In [ ]:
plot_confusion_matrix(y_test, y_predicted)

> Accuracy:  
> Overall correctness of preditions - total correctly predicted vs total predicted
>
> Precision:  
> Correctness of prediction within a predicted class - correctly predicted vs how many predicted to be in the class
>
> Recall:  
> Correctness of prediction within the entire class - correctly predicted vs how many actually in the class
>
> F1-score:  
> Balance of precision and recall (Harmonic mean - bias toward low values to prevent hiding imbalance)

In [ ]:
def display_scores(y_test, y_predicted):
    accuracy = accuracy_score(y_test, y_predicted)
    precision = precision_score(y_test, y_predicted, average='weighted')
    recall = recall_score(y_test, y_predicted, average='weighted')
    f1 = f1_score(y_test, y_predicted, average='weighted')

    print(f"Accuracy: {accuracy * 100:.2f}%")
    print(f"Precision: {precision * 100:.2f}%")
    print(f"Recall: {recall * 100:.2f}%")
    print(f"F1-Score: {f1 * 100:.2f}%")

display_scores(y_test, y_predicted)

In [ ]:
def display_class_scores(y_test, y_predicted):
    precision_per_class = precision_score(y_test, y_predicted, average=None)
    recall_per_class = recall_score(y_test, y_predicted, average=None)
    f1_per_class = f1_score(y_test, y_predicted, average=None)

    metrics = pd.DataFrame({
        "Class": range(len(precision_per_class)),
        "Precision": precision_per_class,
        "Recall": recall_per_class,
        "F1-Score": f1_per_class
    })

    print(metrics)

display_class_scores(y_test, y_predicted);

> Rather than manually displaying scores, there's a built in function ```classification_report```.  

In [ ]:
print(classification_report(y_test, y_predicted, digits=4))

> The next few cells show the affects of scaling, concentrating values around the mean with a standard deviation of 1 
> (StandardScaler), or within 0 and 1 (MinMaxScaler)

In [ ]:
def plot_scatter(title, scaler=None):
    size = 50
    matrix = np.random.uniform(0, 10, (size, size))

    if scaler:
        matrix = scaler.fit_transform(matrix)

    plt.figure(figsize=(6, 3), constrained_layout=True)
    plt.scatter(matrix[:, 0], matrix[:, 1], color=plt.cm.Greens(0.5))

    plt.xlim(-10, 10)
    plt.ylim(-10, 10)

    plt.axhline(y=0, linestyle='--', color=plt.cm.Reds(0.5))
    plt.axvline(x=0, linestyle='--', color=plt.cm.Reds(0.5))

    plt.title(title)
    plt.xlabel("x")
    plt.ylabel("y")
    plt.grid()

    plt.show()

In [ ]:
plot_scatter("Random +/+")

In [ ]:
plot_scatter("Fit transform with StandardScaler", StandardScaler())

In [ ]:
plot_scatter("Fit transform with MinMaxScaler", MinMaxScaler())

> Shows how using different scalers affect the accuracy of the trained model.

In [ ]:
def fit_and_predict(X_train, y_train, X_test, scalers=None, **classifier_params):
    if scalers:
        for scaler in scalers:
            X_train = scaler.fit_transform(X_train)
            X_test = scaler.transform(X_test)

    # Train
    model = LogisticRegression(max_iter=1000, **classifier_params)
    model.fit(X_train, y_train)

    # Evaluate
    y_predicted = model.predict(X_test)

    return y_predicted

In [ ]:
y_predicted = fit_and_predict(X_train, y_train, X_test)

print("No scaler with class_weight=None:")
print(classification_report(y_test, y_predicted, digits=4))
print()

y_predicted = fit_and_predict(X_train, y_train, X_test, class_weight="balanced")

print("No scaler with class_weight='balanced':")
print(classification_report(y_test, y_predicted, digits=4))
print()

y_predicted = fit_and_predict(X_train, y_train, X_test, [StandardScaler()])

print("With StandardScaler:")
print(classification_report(y_test, y_predicted, digits=4))
print()

y_predicted = fit_and_predict(X_train, y_train, X_test, [MinMaxScaler()])
print("With MinMaxScaler:")
print(classification_report(y_test, y_predicted, digits=4))

> Show how well the scaled images match the original.

In [ ]:
def plot_scaled_images(X_train, y_train, digit, scalers, scaled_callback=None):   
    indexes = np.where(y_train == digit)[0]
    X_train_selected =  X_train.iloc[indexes]

    num_images = min(len(indexes), 5)

    _, axes = plt.subplots(len(scalers) + 1, num_images, figsize=(8, 5), constrained_layout=True)

    for index in range(num_images):
        original_image = X_train_selected.iloc[index].values.reshape(28, 28)
        
        axes[0, index].imshow(original_image, cmap=plt.cm.Greys_r)
        axes[0, index].set_title("Original")
        axes[0, index].axis("off")

        for scaler_index in range(len(scalers)):
             scaler = scalers[scaler_index]
             X_train_scaled = scaler.fit_transform(X_train)

             if scaled_callback:
                 X_train_scaled = scaled_callback(X_train_scaled)

             X_train_scaled_selected = X_train_scaled[indexes]

             scaled_image = X_train_scaled_selected[index].reshape(28, 28)

             axes[scaler_index + 1, index].imshow(scaled_image, cmap=plt.cm.Greys_r)
             axes[scaler_index + 1, index].set_title(type(scalers[scaler_index]).__name__)
             axes[scaler_index + 1, index].axis("off")

    plt.show()

In [ ]:
def plot_digit_comparisons():
    scalers = [StandardScaler(), MinMaxScaler()]

    plot_scaled_images(X_train, y_train, 3, scalers)
    plot_scaled_images(X_train, y_train, 5, scalers)
    plot_scaled_images(X_train, y_train, 8, scalers)

plot_digit_comparisons()

> Shows how reducing the number of features using Principal Component Analysis affects the accuracy of the trained model.

In [ ]:
def reduce_dimensions(X_train):
    pca = PCA(n_components=10)

    X_train_scaled = pca.fit_transform(X_train)

    return X_train_scaled

reduce_dimensions(X_train)[0]

In [ ]:
def plot_scaled_images_with_pca(X_train, y_train):
    pca = PCA(n_components=100)

    plot_scaled_images(X_train, y_train, 3, [pca], lambda X_train_scaled: pca.inverse_transform(X_train_scaled))

plot_scaled_images_with_pca(X_train, y_train);

> Shows how using a polynomial function affects the accuracy of the trained model.

In [ ]:
poly = PolynomialFeatures(degree=2, include_bias=False)

X_train_poly = poly.fit_transform(reduce_dimensions(X_train))

X_train_poly

In [ ]:
def fit_and_predict_with_scaler_and_pca_and_polynomial():
    minmax = MinMaxScaler()
    pca = PCA(n_components=100)
    poly = PolynomialFeatures(degree=2, include_bias=False)

    y_predicted = fit_and_predict(X_train, y_train, X_test, [minmax, pca, poly])

    return y_predicted

y_predicted = fit_and_predict_with_scaler_and_pca_and_polynomial()

print("PCA and Polynomial")
print(classification_report(y_test, y_predicted, digits=4))

plot_confusion_matrix(y_test, y_predicted)

plot_misclassified_images(len(y_train), X_test, y_test, y_predicted)

> Rather than using the custom function ```fit_and_predict``` that contains demonstration code, this is just the actual 
> code required.
>
> In ```fit_and_predict``` both the training and prediction was done in the same place with the training data and 
> testing data going through the same transformations. However, this is unrealistic, as once the model is trained, you 
> would normally want to predict individual values without having to retrain. This would make training and prediction 
> two separate processes. The snag would be that when predicting you would need to know the same transformations
> that where applied at training, otherwise you would have a mismatch as get unreliable results at best or, more 
> likely, just get errors.
> 
> Instead of running the scaler, pca, polymonial, and logistic regression, one at a time with the results from one 
> feeding into the next, this can be replaced with a pipeline. The pipeline ensures that training data passes through 
> this process, and is part of the model. Then when the model is used to predict, the same process will be applied,
> ensuring there is no mismatch in scaling, etc. 
>
>  This is the final, minimal code example.

In [ ]:
def train_model(X_train, y_train):
    model = make_pipeline(
        MinMaxScaler(),
        PCA(n_components=100),
        PolynomialFeatures(degree=2, include_bias=False),
        LogisticRegression(max_iter=1000)
    )

    model.fit(X_train, y_train)

    return model

In [ ]:
model = train_model(X_train, y_train)

y_predicted = model.predict(X_test)

print(classification_report(y_test, y_predicted, digits=4))

plot_confusion_matrix(y_test, y_predicted)

> Shows saving the trained model, then loading it and making predictions as a separate step.

In [ ]:
JOBLIB_MODEL_FILENAME="C:\\temp\\mnist_model.joblib"

In [ ]:
def save_model_as_joblib(model, model_filename): joblib.dump(model, model_filename)

def load_joblib_model(model_filename): return joblib.load(model_filename)

In [ ]:
model = train_model(X_train, y_train)

save_model_as_joblib(model, JOBLIB_MODEL_FILENAME)

In [ ]:
model = load_joblib_model(JOBLIB_MODEL_FILENAME);

y_predicted = model.predict(X_test)

print(classification_report(y_test, y_predicted, digits=4))

> For reference only:  
> The following is the full example provided by scikit-learn.  
> This example is self-contained and does not rely on previous cells having been executed.  
>
> https://scikit-learn.org/stable/auto_examples/linear_model/plot_sparse_logistic_regression_mnist.html

In [ ]:
import time

import matplotlib.pyplot as plt
import numpy as np

from sklearn.datasets import fetch_openml
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.utils import check_random_state

def run_scikit_learn_example():
    # Turn down for faster convergence
    t0 = time.time()
    train_samples = 5000

    # Load data from https://www.openml.org/d/554
    X, y = fetch_openml("mnist_784", version=1, return_X_y=True, as_frame=False)

    random_state = check_random_state(0)
    permutation = random_state.permutation(X.shape[0])
    X = X[permutation]
    y = y[permutation]
    X = X.reshape((X.shape[0], -1))

    X_train, X_test, y_train, y_test = train_test_split(
        X, y, train_size=train_samples, test_size=10000
    )

    scaler = StandardScaler()
    X_train = scaler.fit_transform(X_train)
    X_test = scaler.transform(X_test)

    # Turn up tolerance for faster convergence
    clf = LogisticRegression(C=50.0 / train_samples, l1_ratio=1, solver="saga", tol=0.1)
    clf.fit(X_train, y_train)
    sparsity = np.mean(clf.coef_ == 0) * 100
    score = clf.score(X_test, y_test)
    # print('Best C % .4f' % clf.C_)
    print("Sparsity with L1 penalty: %.2f%%" % sparsity)
    print("Test score with L1 penalty: %.4f" % score)

    coef = clf.coef_.copy()
    plt.figure(figsize=(10, 5))
    scale = np.abs(coef).max()
    for i in range(10):
        l1_plot = plt.subplot(2, 5, i + 1)
        l1_plot.imshow(
            coef[i].reshape(28, 28),
            interpolation="nearest",
            cmap=plt.cm.RdBu,
            vmin=-scale,
            vmax=scale,
        )
        l1_plot.set_xticks(())
        l1_plot.set_yticks(())
        l1_plot.set_xlabel(f"Class {i}")
    plt.suptitle("Classification vector for...")

    run_time = time.time() - t0
    print("Example run in %.3f s" % run_time)
    plt.show()

run_scikit_learn_example()

> For curiosity only:  
> The following are 2D and 3D plots of the MNIST dataset by digit.  
> These examples are self-contained and do not rely on previous cells having been executed.  

In [ ]:
import numpy as np

from sklearn.datasets import fetch_openml
from sklearn.preprocessing import MinMaxScaler, PolynomialFeatures
from sklearn.decomposition import PCA
from sklearn.pipeline import make_pipeline

import matplotlib.pyplot as plt

def plot_2d_visualization():
    mnist = fetch_openml('mnist_784', version=1)
    X_train = mnist["data"][:60000]
    y_train = mnist["target"][:60000].astype(int)

    model = make_pipeline(
        MinMaxScaler(),
        PCA(n_components=2),
        PolynomialFeatures(degree=2, include_bias=False)
    )

    Xt = model.fit_transform(X_train)

    fig, ax = plt.subplots(figsize=(20, 12))
    scatter = ax.scatter(Xt[:, 0], Xt[:, 1], c=y_train, cmap='tab10', alpha=0.6, s=4)
    ax.set_title('MNIST Dataset: 2D Visualization')
    ax.get_xaxis().set_visible(False)
    ax.get_yaxis().set_visible(False)
    cbar = fig.colorbar(scatter, ax=ax)
    cbar.set_label('Digit')

    plt.show()

plot_2d_visualization()

In [ ]:
import numpy as np

from sklearn.datasets import fetch_openml
from sklearn.preprocessing import MinMaxScaler, PolynomialFeatures
from sklearn.decomposition import PCA
from sklearn.pipeline import make_pipeline

import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D

def plot_3d_visualization():
    mnist = fetch_openml('mnist_784', version=1)
    X_train = mnist["data"][:60000]
    y_train = mnist["target"][:60000].astype(int)

    model = make_pipeline(
        MinMaxScaler(),
        PCA(n_components=3),
        PolynomialFeatures(degree=2, include_bias=False)
    )

    Xt = model.fit_transform(X_train)

    fig = plt.figure(figsize=(20, 12))
    ax = fig.add_subplot(111, projection='3d')

    scatter = ax.scatter(Xt[:, 0], Xt[:, 1], Xt[:, 2], c=y_train, cmap='tab10', alpha=0.6, s=4)
    ax.set_title('MNIST Dataset: 3D Visualization')
    ax.set_xticklabels([])
    ax.set_yticklabels([])
    ax.set_zticklabels([])

    cbar = fig.colorbar(scatter, ax=ax)
    cbar.set_label('Digit')

    plt.show()

plot_3d_visualization()